# 01. 数据加载与初步浏览

**数据来源**：UCI Online Retail II（跨境电商交易记录，2009-12 ~ 2011-12）

**本 notebook 的目标**：
1. 把 Excel 读入 pandas DataFrame
2. 看清楚数据长什么样：有多少行、哪些列、每列的类型
3. 快速扫一遍异常：缺失值、负数、重复行

**使用方法**：选中下方每一个代码单元格，按 `Shift + Enter` 运行。

## 1. 导入库

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 让 matplotlib 中文正常显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 让 pandas 显示所有列
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print('pandas', pd.__version__)

pandas 3.0.6


## 2. 读入 Excel

数据集是 xlsx 格式，有两个 sheet：`Year 2009-2010` 和 `Year 2010-2011`。先把两张都读进来，然后拼成一张完整表。

In [2]:
data_path = '../data/online_retail_II.xlsx'

# 读两个 sheet（这一步可能慢 1-2 分钟，xlsx 文件大）
df_09_10 = pd.read_excel(data_path, sheet_name='Year 2009-2010')
df_10_11 = pd.read_excel(data_path, sheet_name='Year 2010-2011')

print('2009-2010:', df_09_10.shape)
print('2010-2011:', df_10_11.shape)

2009-2010: (525461, 8)
2010-2011: (541910, 8)


In [3]:
# 把两年数据合并成一张表，方便后续统一分析
df = pd.concat([df_09_10, df_10_11], ignore_index=True)
print('合并后形状:', df.shape)
df.head()

合并后形状: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 3. 字段含义

- `Invoice`：订单号。以 `C` 开头的是退款单。
- `StockCode`：商品编码
- `Description`：商品描述
- `Quantity`：购买数量（退款时为负）
- `InvoiceDate`：下单时间
- `Price`：单价（英镑）
- `Customer ID`：客户编号。空值表示非注册会员。
- `Country`：客户所在国家

## 4. 各列类型与缺失情况

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [5]:
# 每列的缺失数量
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

## 5. 数值字段的描述统计

In [6]:
df[['Quantity', 'Price']].describe()

,Quantity,Price
count,1.067371e+06,1.067371e+06
mean,9.938898e+00,4.649388e+00
std,1.727058e+02,1.235531e+02
min,-8.099500e+04,-5.359436e+04
25%,1.000000e+00,1.250000e+00
50%,3.000000e+00,2.100000e+00
75%,1.000000e+01,4.150000e+00
max,8.099500e+04,3.897000e+04


注意观察：
- `Quantity` 的 min 是不是负数？负数来源于退款（Invoice 以 C 开头）。
- `Price` 的 min 是不是 0 或负数？0 或负价说明是异常/调整单，后面要清理掉。

## 6. 其他基本面

In [7]:
# 时间跨度
print('最早订单:', df['InvoiceDate'].min())
print('最晚订单:', df['InvoiceDate'].max())

# 唯一值计数
print('\n唯一订单数:', df['Invoice'].nunique())
print('唯一商品数:', df['StockCode'].nunique())
print('唯一客户数:', df['Customer ID'].nunique())
print('涉及国家数:', df['Country'].nunique())

最早订单: 2009-12-01 07:45:00
最晚订单: 2011-12-09 12:50:00

唯一订单数: 53628
唯一商品数: 5305
唯一客户数: 5942
涉及国家数: 43


In [8]:
# TOP 10 国家（按订单数）
df['Country'].value_counts().head(10)

Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64

In [9]:
# 退款单占比：Invoice 以 C 开头的记录
n_refund = df['Invoice'].astype(str).str.startswith('C').sum()
print(f'退款记录数: {n_refund}')
print(f'占总记录比例: {n_refund / len(df) * 100:.2f}%')

退款记录数: 19494
占总记录比例: 1.83%


## 7. 把 DataFrame 存成高效格式（可选）

下一个 notebook 会继续用这份数据。xlsx 读入慢，这里存成 parquet 格式，后面秒读。

In [10]:
# 用 pickle 保存（最简单，不需要额外依赖）
df.to_pickle('../data/retail_raw.pkl')
print('已保存 ../data/retail_raw.pkl，下次直接 pd.read_pickle 读取即可')

已保存 ../data/retail_raw.pkl，下次直接 pd.read_pickle 读取即可


---

**本 notebook 到此结束。**

下一步会在 `02_clean.ipynb` 做数据清洗（处理缺失客户 ID、剔除退款负数、过滤 0/负价、去重）。